<a href="https://colab.research.google.com/github/ayachraiet88-crypto/breast-cancer-predictor/blob/main/Assistant_Sant%C3%A9_%E2%80%94_Chatbot_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-community langchain-openai chromadb sentence-transformers unstructured


In [2]:
import os


In [3]:
from google.colab import userdata


In [4]:
from google.colab import files


In [5]:
import os

In [6]:
os.makedirs("documents", exist_ok=True)


In [7]:

uploaded = files.upload()

Saving cold_vs_flu.md to cold_vs_flu.md
Saving first_aid_basics.txt to first_aid_basics.txt
Saving health_faq.txt to health_faq.txt
Saving nutrition_basics.txt to nutrition_basics.txt
Saving sleep_hygiene.md to sleep_hygiene.md


In [8]:
for fname in uploaded.keys():
    os.rename(fname, f"documents/{fname}")

In [9]:
print("Fichiers dans documents/:", os.listdir("documents"))

Fichiers dans documents/: ['first_aid_basics.txt', 'sleep_hygiene.md', 'cold_vs_flu.md', 'nutrition_basics.txt', 'health_faq.txt']


In [10]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader


/tmp/ipykernel_980/707324743.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader


In [11]:
import os

In [12]:
def load_documents(folder="documents"):
    docs = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        try:
            if fname.endswith(".txt"):
                loader = TextLoader(path, encoding="utf-8")
            elif fname.endswith(".md"):
                loader = UnstructuredMarkdownLoader(path)
            else:
                print(f"Format non géré, ignoré : {fname}")
                continue
            docs.extend(loader.load())
            print(f"Chargé : {fname}")
        except Exception as e:
            print(f"Erreur avec {fname} : {e}")
    return docs

In [13]:
raw_documents = load_documents("documents")


Chargé : first_aid_basics.txt
Chargé : sleep_hygiene.md
Chargé : cold_vs_flu.md
Chargé : nutrition_basics.txt
Chargé : health_faq.txt


In [14]:
print(f"\nTotal : {len(raw_documents)} document(s) chargé(s)")


Total : 5 document(s) chargé(s)


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

In [17]:
chunks = text_splitter.split_documents(raw_documents)


In [18]:
print(f"{len(raw_documents)} documents découpés en {len(chunks)} chunks")


5 documents découpés en 24 chunks


In [19]:
print("\n--- Exemple de chunk ---\n")
print(chunks[0].page_content)


--- Exemple de chunk ---

BASIC FIRST AID - GENERAL REFERENCE

1. Minor Cuts and Scrapes
- Clean the wound with running water.
- Apply gentle pressure with a clean cloth to stop any bleeding.
- Apply an antiseptic if available, then cover with a sterile bandage.
- Seek medical attention if bleeding does not stop after 10 minutes of pressure, or if the wound is deep.


In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings


In [21]:
from langchain_community.vectorstores import Chroma


In [22]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

/tmp/ipykernel_980/209708696.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)


In [24]:
print(f"Base ChromaDB recréée avec {vectorstore._collection.count()} chunks embeddés")

Base ChromaDB recréée avec 24 chunks embeddés


In [25]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # on récupère les 3 chunks les plus proches de la question
)



In [26]:
query = "Que faire en cas de coupure ?"
results = retriever.invoke(query)

In [27]:
print(f"Question : {query}\n")


Question : Que faire en cas de coupure ?



In [28]:
print(f"{len(results)} chunks trouvés :\n")


3 chunks trouvés :



In [29]:
for i, doc in enumerate(results):
    print(f"--- Résultat {i+1} (source : {doc.metadata.get('source', 'inconnue')}) ---")
    print(doc.page_content)
    print()

--- Résultat 1 (source : documents/cold_vs_flu.md) ---
General Self-Care Tips

Rest and stay hydrated.

Over-the-counter remedies may relieve symptoms such as congestion or fever, according to product instructions.

Wash hands frequently to avoid spreading illness to others.

When to See a Doctor

Seek medical care for difficulty breathing, chest pain, persistent high fever, symptoms that improve then worsen, or if the person is in a high-risk group (young children, elderly, pregnant, or immunocompromised individuals).

--- Résultat 2 (source : documents/first_aid_basics.txt) ---
BASIC FIRST AID - GENERAL REFERENCE

1. Minor Cuts and Scrapes
- Clean the wound with running water.
- Apply gentle pressure with a clean cloth to stop any bleeding.
- Apply an antiseptic if available, then cover with a sterile bandage.
- Seek medical attention if bleeding does not stop after 10 minutes of pressure, or if the wound is deep.

--- Résultat 3 (source : documents/first_aid_basics.txt) ---
5. Faint

In [30]:
from langchain_openai import ChatOpenAI


In [31]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant santé qui répond aux questions en te basant UNIQUEMENT sur le contexte fourni ci-dessous.
Si l'information n'est pas présente dans le contexte, dis-le clairement au lieu d'inventer une réponse.
Réponds dans la même langue que la question.

Contexte :
{context}

Question : {question}

Réponse :
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [34]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 1.6 MB/s eta 0:00:00


In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
import os

In [36]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


In [57]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",

)

In [58]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant santé qui répond aux questions en te basant UNIQUEMENT sur le contexte fourni ci-dessous.
Si l'information n'est pas présente dans le contexte, dis-le clairement au lieu d'inventer une réponse.
Réponds dans la même langue que la question.

Contexte :
{context}

Question : {question}

Réponse :
""")

In [59]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [60]:
question = "Que faire en cas de coupure ?"
reponse = rag_chain.invoke(question)
print(reponse)

En cas de coupure mineure, voici la procédure à suivre :

*   Nettoyez la plaie avec de l'eau courante.
*   Appliquez une pression douce avec un linge propre pour arrêter tout saignement.
*   Appliquez un antiseptique si vous en avez, puis couvrez avec un pansement stérile.
*   Consultez un médecin si le saignement ne s'arrête pas après 10 minutes de pression ou si la coupure est profonde.


In [61]:
def poser_question(question):
    reponse = rag_chain.invoke(question)
    print(f"❓ Question : {question}")
    print(f"💬 Réponse : {reponse}\n")
    print("-" * 60)


In [62]:
questions_test = [
    "Que faire en cas de coupure ?",
    "Combien d'heures de sommeil recommandées par nuit ?",
    "Quelle est la différence entre un rhume et la grippe ?",
    "Quels aliments privilégier pour une alimentation équilibrée ?",
    "Que faire si quelqu'un s'étouffe ?",
]

In [63]:
for q in questions_test:
    poser_question(q)

❓ Question : Que faire en cas de coupure ?
💬 Réponse : En cas de coupure mineure, voici les étapes à suivre selon le document :

*   Nettoyez la plaie à l'eau courante.
*   Appliquez une pression douce avec un linge propre pour arrêter tout saignement.
*   Appliquez un antiseptique si disponible, puis couvrez avec un pansement stérile.
*   Consultez un médecin si le saignement ne s'arrête pas après 10 minutes de pression ou si la coupure est profonde.

------------------------------------------------------------
❓ Question : Combien d'heures de sommeil recommandées par nuit ?
💬 Réponse : La durée de sommeil recommandée par nuit dépend de l'âge :

*   **Adultes (18-64 ans) :** 7 à 9 heures
*   **Teenagers (14-17 ans) :** 8 à 10 heures
*   **Personnes âgées (65 ans et plus) :** 7 à 8 heures

------------------------------------------------------------
❓ Question : Quelle est la différence entre un rhume et la grippe ?
💬 Réponse : Le rhume et la grippe sont deux maladies respiratoires cau

In [64]:
print("\n=== TEST HORS-CONTEXTE (vérification anti-hallucination) ===\n")
question_hors_sujet = "Quelle est la capitale de la France ?"
poser_question(question_hors_sujet)


=== TEST HORS-CONTEXTE (vérification anti-hallucination) ===

❓ Question : Quelle est la capitale de la France ?
💬 Réponse : Réponse : L'information n'est pas présente dans le contexte fourni.

------------------------------------------------------------


In [65]:
debug_results = retriever.invoke("Que faire si quelqu'un s'étouffe ?")
for i, doc in enumerate(debug_results):
    print(f"--- Résultat {i+1} (source: {doc.metadata.get('source')}) ---")
    print(doc.page_content[:150], "...\n")

--- Résultat 1 (source: documents/first_aid_basics.txt) ---
5. Fainting
- Lay the person flat and elevate their legs if possible.
- Loosen tight clothing around the neck.
- Ensure fresh air and check responsive ...

--- Résultat 2 (source: documents/cold_vs_flu.md) ---
General Self-Care Tips

Rest and stay hydrated.

Over-the-counter remedies may relieve symptoms such as congestion or fever, according to product inst ...

--- Résultat 3 (source: documents/first_aid_basics.txt) ---
6. When to Call Emergency Services
Always call local emergency services for: severe bleeding, difficulty breathing, chest pain, signs of stroke (face  ...



In [66]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [67]:
!pip install -q gradio

In [68]:
import gradio as gr


In [69]:

def chatbot_interface(question, history):
    if not question.strip():
        return "Merci de poser une question."
    reponse = rag_chain.invoke(question)
    return reponse

In [71]:
demo = gr.ChatInterface(
    fn=chatbot_interface,
    title=" Assistant Santé — Chatbot RAG",
    description=(
        "Posez une question sur : premiers secours, sommeil, nutrition, "
        "rhume vs grippe, ou consultez la FAQ santé. "
        "Les réponses sont générées à partir de documents de référence via RAG (LangChain + ChromaDB)."
    ),
    examples=[
        "Que faire en cas de coupure ?",
        "Combien d'heures de sommeil recommandées par nuit ?",
        "Quelle est la différence entre un rhume et la grippe ?",
        "Quels aliments privilégier pour une alimentation équilibrée ?",
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a16ac353af5925284.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
